# **Atividade - Classificação Clássica e MLP com PyTorch**

**Observação:** A parte de redes neurais (Exercício 3) deve ser implementada obrigatoriamente com **PyTorch**.

*   **Nome**: Giovanni Daldegan
*   **Matrícula**: 232002520

URL do diretório do projeto no repositório hospedeiro no GitHub: \
https://github.com/GiovanniDaldegan/cic-unb/tree/main/26.1/IIA/trabalho

---


# **Diagnóstico de Diabetes com Aprendizado de Máquina**

Nesta atividade, vamos trabalhar com um problema aplicado de **classificação binária**: prever se uma pessoa possui ou não diabetes com base em um conjunto de variáveis clínicas.

[Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)

---

## **Contexto**

- O dataset utilizado é o **Pima Indians Diabetes Dataset**, coletado originalmente pelo Instituto Nacional de Diabetes e Doenças Digestivas e Renais dos Estados Unidos.
- Ele contém registros de mulheres com pelo menos 21 anos de idade da população Pima, um grupo étnico nativo norte-americano com alta incidência de diabetes tipo 2.

## **Objetivo**

O objetivo é explorar diferentes técnicas de aprendizado de máquina — desde modelos clássicos até redes neurais — para prever a presença de diabetes a partir de atributos fisiológicos e laboratoriais.

## **Variáveis de entrada**

Cada observação contém os seguintes atributos:

1. **Pregnancies**, number of times pregnant: Variável discreta.
2. **Glucose**, plasma glucose concentration after 2 hours in an oral glucose tolerance test: Variável contínua.
3. **BloodPressure**, diastolic blood pressure, in mm Hg: Variável contínua.
4. **SkinThickness**, triceps skin fold thickness, in mm: Variável contínua.
5. **Insulin**, 2-hour serum insulin, in μU/mL: Variável contínua.
6. **BMI**, body mass index, weight in kg/(height in m)²: Variável contínua.
7. **DiabetesPedigreeFunction**, family history function: Variável contínua.
8. **Age**, in years: Variável discreta.

## **Variáveis de saída (Target)**

- **Outcome = 1**: Diabetic
- **Outcome = 0**: Non-diabetic


# **Exercício 1 - Preparação dos dados**

1. Realize tratamento e limpeza de dados caso necessário para treinamento do modelo
2. Crie a matriz de correlação das features e extraia pelo menos um insight estatístico visível na matriz


#### Código de apoio — Carregamento do dataset
Execute a célula abaixo para carregar o dataset. Use este dataframe como ponto de partida.


In [ ]:
import pandas as pd

# URL do dataset
url = "https://raw.githubusercontent.com/pcbrom/perceptron-mlp-cnn/refs/heads/main/data/diabetes.csv"

# Carregar o dataset
starting_df = pd.read_csv(url)
starting_df.head()

#### Seu código — Exercício 1


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def clean_dataframe(df:pd.DataFrame) -> dict:
    starting_count = curr_count = last_count = df.shape[0]
    dropped_data_na = dict()

    # remover duplicatas
    df.drop_duplicates()
    curr_count = df.shape[0]
    dropped_dup = last_count - curr_count
    last_count = curr_count

    # remover entradas com dados faltantes ou impossíveis (valores 0 em colunas estritamente positivas)
    df.dropna()

    aux_last_count = last_count

    for col in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "Age"]:
        df = df[df[col] > 0]
        curr_count = df.shape[0]

        # atualizar dicionário de entradas descartadas por coluna
        if aux_last_count - curr_count != 0:
            if col not in dropped_data_na.keys():
                dropped_data_na[col] = 0

            dropped_data_na[col] += aux_last_count - curr_count
        
        aux_last_count = curr_count

    curr_count = df.shape[0]
    dropped_na = last_count - curr_count
    last_count = curr_count

    return df, {
        "starting_count" : starting_count,
        "final_count" : curr_count,
        "dropped" : {
            "total" : starting_count - curr_count,
            "duplicates" : dropped_dup,
            "na" : dropped_na,
            "data_na" : dropped_data_na
        }
    }

def corr_matrix_df(df:pd.DataFrame, title:str, message:str=None):
    corr_mat = df.corr()

    print(message)
    plt.figure(figsize=(10, 8))
    plt.title(title)
    sns.heatmap(corr_mat, annot=True, cmap='coolwarm', center=0, # 
                square=True, linewidths=0.5, vmin=-1.0, vmax=1.0)
    plt.tight_layout()
    plt.show()

df, pre_training_summary = clean_dataframe(starting_df)

print(f"""===========================
Base de dados pré-filtragem
===========================

Total de entradas: {pre_training_summary["starting_count"]}
URL fonte: {url}


===========================
Base de dados pós-filtragem
===========================

Total de entradas: {pre_training_summary["final_count"]}
Entradas desconsideradas: {pre_training_summary["dropped"]["total"]}
- Duplicatas: {pre_training_summary["dropped"]["duplicates"]}
- Dados faltando: {pre_training_summary["dropped"]["na"]}
""")

for k, v in pre_training_summary["dropped"]["data_na"].items():
    print(f"    - {k}: {v}")
print()

corr_matrix_df(starting_df, "Correlação entre atributos (dataset original)", "Matriz de correlação do dataset original:")
print()

corr_matrix_df(df, "Correlação entre atributos (dataset filtrado)", "Matriz de correlação do dataset pós-filtragem:")

#### Respostas Exercício 1:

Com a base de dados salva na variável `df`, é aplicada uma limpeza no *dataset*.

A limpeza remove todas entradas que se encaixam nos critérios:
- entradas duplicadas;
- entradas com valores faltando e/ou impossíveis (no caso, medições nulas que deveriam ser estritamente maiores que 0).

Realizando essa filtragem, entre as 768 entradas iniciais, apenas 392 não apresentavam nenhuma inconsistência, enquanto as 376 restantes foram descartadas para evitar a consideração de dados seguramente imprecisos.

<br>

Com o *dataset* limpo, é obtida a matriz de correlação entre seus atributos.

Há ocorrências de atributos com correlação um pouco alta, mas não total. Os maiores valores de correlação variam de 0.52 a 0.68, porém não são altos o suficiente para que seja necessário remover um desses atributos. Ainda mais porque cada par de atributos tem valores de correlação diferentes em relação aos demais atributos.

<br>

Um detalhe considerável é que alguns os valores de correlação do *dataset* pós-filtragem são maiores que os valores do *dataset* inicial, como é possível notar nas respectivas matrizes de correlação.

Porém, espera-se que esse aumento geral não seja suficiente para induzir um caso de *overfitting* dos modelos a serem treinados ou na formação de viéses fortes, pois apenas 4 valores de correlação passam de 0.5 e a grande maioria continua mais próxima de 0.

## **Exercício 2 — Comparação de Modelos de Classificação: KNN vs Regressão Logística**

### Objetivo
Treinar e comparar dois modelos de classificação supervisionada usando **scikit-learn**:
- **KNeighborsClassifier** (K-Nearest Neighbors)
- **LogisticRegression** (Regressão Logística)

O foco é **encontrar a melhor combinação de hiperparâmetros** para cada modelo e **analisar qual apresenta melhor desempenho** nos conjuntos de treino e teste.

---

### Contexto dos Modelos

- **KNN**: Um algoritmo baseado em instâncias. Para classificar um novo dado, ele encontra os **K vizinhos mais próximos** no conjunto de treino e faz uma "votação" — a classe mais frequente entre os vizinhos é a predição.
- **Regressão Logística**: Um modelo linear que aprende **pesos** para cada variável de entrada e usa uma função **sigmoide** para gerar a probabilidade de cada classe. Pode ser entendido como um **neurônio único** — uma prévia do que veremos no Exercício 3 com redes neurais!

---

### Instruções

1. **Divisão dos dados**
   - Utilize o mesmo dataset do Exercício 1 (já limpo e tratado).
   - Divida os dados em **80% para treino** e **20% para teste**, de forma **estratificada** para manter a proporção de classes.
   - Use `train_test_split` do **scikit-learn** com `random_state=42`.

2. **Treinamento dos modelos**
   - Crie dois modelos e os treine utilizando o **conjunto de treino**:
     - `KNeighborsClassifier`
     - `LogisticRegression`

3. **Busca de hiperparâmetros**
   - Utilize **GridSearchCV** para encontrar a melhor combinação de parâmetros para cada modelo.
   - Varie os seguintes parâmetros:
     - **KNN** → `n_neighbors` (ex: 3, 5, 7, 9, 11), `weights` ('uniform', 'distance'), `metric` ('euclidean', 'manhattan').
     - **Regressão Logística** → `C` (ex: 0.01, 0.1, 1, 10, 100), `penalty` ('l1', 'l2'), `solver` ('liblinear').

4. **Avaliação**
   - Avalie os **melhores modelos** (após o GridSearch) usando o **conjunto de teste**.
   - Calcule as métricas:
     - **Acurácia**
     - **F1-Score**

5. **Apresentação dos resultados**
   - Monte uma **tabela Markdown** comparando os modelos (veja o modelo na seção de Respostas abaixo).

6. **Análise dos resultados**
   - Responda:
     - a) Qual modelo obteve melhor desempenho? Por quê?
     - b) Houve **overfitting** em algum dos modelos? Como você justifica?
     - c) Qual é a diferença fundamental entre como o KNN e a Regressão Logística fazem suas predições? Qual abordagem é mais semelhante ao funcionamento de uma rede neural?



#### Seu código — Exercício 2


In [ ]:
##### Seu código aqui #####



#### Respostas Exercício 2:

**Tabela de resultados** (preencha com seus valores):

| Modelo              | Conjunto | Acurácia | F1-Score |
|:--------------------|:---------|----------:|----------:|
| KNN                 | Treino   | 0.0000   | 0.0000   |
| KNN                 | Teste    | 0.0000   | 0.0000   |
| Regressão Logística | Treino   | 0.0000   | 0.0000   |
| Regressão Logística | Teste    | 0.0000   | 0.0000   |

- a)

- b)

- c)


# **Exercício 3 - Construção e Exploração de MLP com PyTorch**

Neste exercício, você vai construir, treinar e comparar diferentes arquiteturas de MLP (Multi-Layer Perceptron) para resolver o mesmo problema de classificação dos exercícios anteriores.

**O uso de PyTorch é obrigatório para resolução deste exercício.**

---

### Instruções

1. **Divisão dos dados**
   - Faça a divisão dos dados:
     - 70% para treino
     - 15% para validação
     - 15% para teste

2. **Treinamento de múltiplas arquiteturas**
   - Treine **pelo menos 3 arquiteturas diferentes** de MLP, variando:
     - Número de camadas ocultas (ex: 1, 2, 3 camadas)
     - Número de neurônios por camada (ex: 16, 32, 64, 128)
     - Função de ativação (ex: ReLU, Tanh, Sigmoid)
   - Para cada arquitetura, utilize o mesmo otimizador e função de perda.

3. **Implementação de Early Stopping**
   - Implemente **early stopping** com base na loss de validação.
   - O treinamento deve parar automaticamente quando a loss de validação **não melhorar** por um número definido de épocas consecutivas (patience).
   - Dica: guarde os pesos do melhor modelo durante o treino.

4. **Avaliação de cada arquitetura**
   - Para **cada** arquitetura treinada, apresente:
     - Acurácia no conjunto de teste
     - Matriz de confusão
     - Relatório de classificação

5. **Tabela comparativa**
   - Monte uma tabela Markdown comparando as arquiteturas (veja o modelo na seção de Respostas abaixo).

6. **Curvas de aprendizado**
   - Plote a **loss de treino** e a **loss de validação** ao longo das épocas para cada arquitetura.
   - Identifique visualmente em qual ponto o early stopping atuou.

7. **Análise dos resultados**
   - Responda:
     - a) O early stopping foi ativado em todas as arquiteturas? Em que época, em média? O que isso indica sobre o treinamento?
     - b) Qual arquitetura obteve a melhor generalização (melhor desempenho no teste)? Ela era a mais complexa?
     - c) Compare as curvas de aprendizado. Houve sinais de overfitting em alguma arquitetura? Como o early stopping ajudou (ou não) a mitigar isso?
     - d) A Regressão Logística do Exercício 2 pode ser vista como um caso especial de MLP (sem camadas ocultas). Comparando os resultados, as camadas adicionais trouxeram benefício significativo para este dataset? Reflita.


#### Código de apoio — Exemplo de Early Stopping
A célula abaixo mostra a lógica básica de early stopping. Adapte e integre ao seu loop de treinamento.


In [ ]:
# === Exemplo de lógica de Early Stopping ===
# Adapte este trecho ao seu loop de treinamento

patience = 10
best_val_loss = float('inf')
counter = 0

for epoch in range(max_epochs):
    # ... treinar ...
    # ... calcular val_loss ...

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        # salvar pesos do melhor modelo
        # torch.save(model.state_dict(), 'best_model.pt')
    else:
        counter += 1
        if counter >= patience:
            print(f'Early stopping na época {epoch}')
            break


#### Código de apoio — Métricas de avaliação
Use as funções abaixo para avaliar cada arquitetura treinada no conjunto de teste.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def avaliar_modelo(y_true_test, y_pred_test, titulo="Modelo"):
    # --- Acurácia
    acc_test = accuracy_score(y_true_test, y_pred_test)
    print(f"Acurácia no conjunto de teste ({titulo}): {acc_test:.4f}")

    # --- Matriz de confusão
    cm = confusion_matrix(y_true_test, y_pred_test)
    plt.figure(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Matriz de Confusão - {titulo}')
    plt.xlabel('Classe Predita')
    plt.ylabel('Classe Real')
    plt.tight_layout()
    plt.show()

    # --- Relatório de classificação
    print("\nRelatório de Classificação:")
    print(classification_report(y_true_test, y_pred_test, digits=4, zero_division=1))


#### Seu código — Exercício 3


In [ ]:
##### Seu código aqui #####



#### Respostas Exercício 3:

**Tabela comparativa das arquiteturas** (preencha com seus valores):

| Arquitetura | Camadas | Neurônios      | Ativação | Época Parada | Acurácia (Teste) | F1-Score (Teste) |
|:------------|:--------|:---------------|:---------|:-------------|:----------------:|:----------------:|
| MLP 1       | 1       | [32]           | ReLU     | --           | 0.0000           | 0.0000           |
| MLP 2       | 2       | [64, 32]       | ReLU     | --           | 0.0000           | 0.0000           |
| MLP 3       | 3       | [128, 64, 32]  | Tanh     | --           | 0.0000           | 0.0000           |

- a)

- b)

- c)

- d)


## QA

1. **No dataset de diabetes, as classes são desbalanceadas (~65% não-diabéticos vs ~35% diabéticos). Como esse desbalanceamento afeta a interpretação da acurácia? Que métricas alternativas seriam mais adequadas para avaliar o modelo nesse cenário?**

2. **Explique a diferença entre o conjunto de validação e o conjunto de teste. Por que é um erro ajustar hiperparâmetros usando o conjunto de teste?**

3. **Um modelo de classificação apresenta acurácia de 100% no conjunto de treino, mas apenas 58% no conjunto de teste. O que está acontecendo? Cite pelo menos duas técnicas que poderiam ser usadas para resolver esse problema.**

4. **Durante o treinamento de uma rede neural profunda, as camadas iniciais deixam de atualizar seus pesos, apresentando gradientes próximos de zero. Que fenômeno está ocorrendo e quais estratégias podem ser aplicadas para solucioná-lo?**

5. **Compare o funcionamento do KNN com o de uma rede neural MLP. Em que situações práticas cada um seria mais vantajoso? Considere aspectos como volume de dados, dimensionalidade e tempo de inferência.**



### Respostas QA

- 1)

- 2)

- 3)

- 4)

- 5)


## Referências

**M. Guimarães, Vítor**. *Classificação: Dos Modelos Clássicos ao MLP*. Universidade de Brasília, Brasília-DF. 27 de maio, 2026.

https://medium.com/@abdallahashraf90x/all-you-need-to-know-about-correlation-for-machine-learning-e249fec292e9

**Xie, S., Zhang, Y., Lv, D. et al**. *A new improved maximal relevance and minimal redundancy method based on feature subset*. J Supercomput 79, 3157–3180 (2023). https://doi.org/10.1007/s11227-022-04763-2
